# Group 9: Disha's data and training (Colab)

**Before you start**
1. Upload `group9_disha.zip` to **My Drive** (not inside a folder). Do not unzip it.
2. In Colab: **Runtime → Change runtime type → T4 GPU → Save**.
3. Run the cells **in order**, from top to bottom (Shift + Enter).

**Every time you come back:** turn on the GPU again, then run cells 1 to 4, then jump to the step you need.
Everything is saved in your Drive folder `group9`, so a disconnect loses nothing.

## 1. Check the GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU ready:", torch.cuda.get_device_name(0))
else:
    print("NO GPU! Go to Runtime -> Change runtime type -> T4 GPU -> Save, then run this cell again.")

## 2. Connect Google Drive
A pop-up asks for permission: click **Connect to Google Drive** and allow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Unzip the code (first time only) and go into the folder
Leave `UPDATE_CODE = False`. Set it to `True` only when you upload a newer zip; your results and splits are never in the zip, so they are kept.

In [ ]:
import os, subprocess, json
from IPython.display import Image, display
UPDATE_CODE = False

ZIP = "/content/drive/MyDrive/group9_disha.zip"
REPO = "/content/drive/MyDrive/group9"
if not os.path.exists(REPO) or UPDATE_CODE:
    assert os.path.exists(ZIP), f"Upload group9_disha.zip to My Drive first (looked for {ZIP})"
    subprocess.run(["unzip", "-q", "-o", ZIP, "-d", "/content/drive/MyDrive/"], check=True)
    print("Code unzipped.")
os.chdir(REPO)
print("Working in:", os.getcwd())
print(sorted(os.listdir(".")))

## 4. Install packages
Colab already has PyTorch; only two packages are missing.

In [ ]:
!pip install -q datasets imagehash
print("Packages ready.")

---
## STEP 1: Data audit and fixed split (run ONCE, tonight)
Downloads the dataset (about 1.2 GB) and takes about 5 to 15 minutes.
Expected: every count says **PASS**, 0 unreadable images, split **3551 / 444 / 444**.
If the split already exists, this cell skips itself.

In [ ]:
if os.path.exists("configs/splits/product_split.csv"):
    print("The split already exists, so step 1 is done. (The split is made once and never remade.)")
else:
    !bash scripts/01_data.sh

### Check the data results
These numbers go into `docs/dataset_card.md`.

In [ ]:
a = json.load(open("results/data_audit/summary.json"))
s = json.load(open("configs/splits/split_summary.json"))
print("Domains found:", a["domain_names_found"])
for k, v in a["count_checks"].items():
    print(f"  {k:<11} {v['found']:>6} (expected {v['expected']})  {v['status']}")
print("Classes:", a["classes"]["num_classes"], "| 'Clock' class present:", a["classes"]["has_class_named_clock"])
print("Unreadable images:", a["unreadable_images"]["count"])
print("Product near-duplicates:", a["product_near_duplicates"])
print("Product vs Real World near-duplicate pairs:", a["cross_domain_near_duplicates"]["pairs"])
print("Flagged samples to review:", a["flagged_samples"]["count"])
print("\nSplit:")
for k, v in s["checks"]["counts"].items():
    print(f"  {k:<5} {v['found']:>5} (expected {v['expected']})  {v['status']}")
print("Duplicate groups crossing splits:", s["checks"]["duplicate_groups_crossing_splits"])
for seed, b in s["budgets"].items():
    print(f"  budgets seed {seed}: {b['sizes']}  nested={b['nested']}")
display(Image("results/figures/product_class_distribution.png", width=900))
display(Image("results/figures/examples_product_vs_realworld.png", width=900))

**Now do by hand:** open the folder icon on the left → `drive/MyDrive/group9/outputs/audit/` and look at the near-duplicate pairs and examples
(wrong labels? identifiable people?). Write what you find in the dataset card.
Then commit `configs/splits/` and `results/data_audit/` to the team GitHub repo **by Nov 7**.

---
## STEP 2: Baseline linear probe (Nov 7)
Trains the linear probe (100% labels, seed 42), saves Product-validation predictions for Shruti, and draws its loss curve. About 10 to 20 minutes.

In [ ]:
!bash scripts/02_baseline.sh

In [ ]:
display(Image("results/figures/loss_curve_lp_b100_s42.png", width=600))
print("Predictions for Shruti are in outputs/predictions/")
print(sorted(os.listdir("outputs/predictions")))

---
## STEP 2b: One fine-tuning run for the progress report (Nov 7 to 8)
Gives the training/validation loss curve for the progress report, and saved epochs Ruchika can test WeightWatcher on.
It uses lr 1e-4 because the pilot has not run yet. About 15 to 25 minutes.

In [ ]:
!python -m src.train.train --mode finetune --budget 100 --seed 42 --lr 1e-4 --run-name progress_ft_b100_s42 --resume
!python -m src.train.report_runs --runs progress_ft_b100_s42
display(Image("results/figures/loss_curve_progress_ft_b100_s42.png", width=600))

---
## STEP 3: Learning-rate pilot (Nov 9)
Two fine-tuning runs (lr 1e-4 and 3e-4), chosen on Product validation loss only. About 30 to 50 minutes.
If it disconnects, run cells 1 to 4 and this cell again: it continues.

In [ ]:
!bash scripts/03_lr_pilot.sh
print(open("configs/lr_choice.json").read())

---
## STEP 4: All 24 runs (Nov 9 to 15)
Order: 100% → 10% → 25% → 50%. Safe to re-run after any disconnect: finished runs are skipped.

⚠️ **Storage:** all 12 fine-tuning runs need about 17 GB, but free Google Drive has 15 GB.
Check the space cell below first. To run part of the budgets, set `BUDGETS`, e.g. `"100 10"`.

In [ ]:
BUDGETS = "100 10 25 50"   # e.g. "100" for the minimum viable project only
!bash scripts/04_train_all.sh --budgets {BUDGETS}

## Storage check (run any time)

In [ ]:
!du -sh checkpoints 2>/dev/null || echo "no checkpoints yet"
!df -h /content/drive | tail -1

## Training-time table (for your report section)

In [ ]:
import pandas as pd
!python -m src.train.report_runs
pd.read_csv("results/training_time_summary.csv")